### Step 3: Feature Engineering

Create features for modeling

In [ ]:
import pandas as pd
import numpy as np

CLEANED_FILEPATH = "../data/002_cleaned/1s_power_load.parquet"

def feature_engineered_filepath(res: str) -> str:
    return f"../data/003_feature_engineered/{res}_power_load.parquet"

RESAMPLE_RESOLUTIONS = ["1min", "5min", "10min"]

LAG_INTERVALS = {
    "1min": 1,
    "5min": 5,
    "15min": 15,
    "30min": 30,
    "1hour": 60,
    "6hour": 360,
    "12hour": 720,
    "1day": 1440,
    "1week": 10080
}
WINDOW_INTERVALS = [5, 15, 30, 60]
VAL_DAYS  = 3
TEST_DAYS = 3
CATEGORICAL_COLS = ["workday", "time_of_day"]
DROP_COLS        = ["timestamp", "load"]

In [2]:
clean_df = pd.read_parquet(CLEANED_FILEPATH)
print(clean_df.shape)
clean_df.head()

(2678400, 3)


,timestamp,load,day_class
0,2025-11-28 00:00:00,1925.0,full
1,2025-11-28 00:00:01,1922.0,full
2,2025-11-28 00:00:02,1921.0,full
3,2025-11-28 00:00:03,1921.0,full
4,2025-11-28 00:00:04,1926.0,full


**Resample data per resolution**:

Our data is in 1-second intervals. We want to resmaple the data to higher resolution intervals for better predictions, such as 30 sec, 1 min, 2 min, 5 min, etc

| **Resolution** | **Description** |
| --- | --- |
| `30s` | 30-second intervals |
| `1min` | 1-minute intervals |
| `2min` | 2-minute intervals |
| `5min` | 5-minute intervals |
| `10min` | 10-minute intervals |
| `15min` | 15-minute intervals |

Each resample of resolution calculates the average of `load` for that resolution

---

**1. Temporal Features**:

Features relating to the time and day

| Column       | Type    | Description |
|-------------|---------|-------------|
| `second` | int | Second of the minute (0,1,...,60) |
| `minute` | int | Minute of the hour (0,1,...,60) |
| `hour` | int | Hour of the day (0,1,...,23) |
| `time_of_day` | string | Time of the day (Morning, Afternoon, Evening, Night)|
| `day`         | int     | Day of month (1–31) |
| `weekday`     | int     | Day of week (0=Mon … 6=Sun) |
| `is_weekend`  | int    | 1 if Saturday or Sunday, 0 otherwise |
| `month`       | int     | Month of year (1=Jan, 2=Feb, ... 12=Dec) |

---

**2. Business Features**:

Features related to the business operations:

| Column | Description |
|--------|-------------|
| `workday` | Type of workday (`full`, `half`, `none`) |

---

**3. Historic Load Features**

Features describe how power load has behaved in the past, across three dimensions:

**Snapshot** — what load was at a specific point in the past

| Column | Description |
|--------|-------------|
| `lag_X` | Load value exactly X intervals ago |

**Window Statistics** — summary of load behavior over a recent period

| Column | Description |
|--------|-------------|
| `rolling_mean_X` | Average load over the past X intervals |
| `rolling_std_X` | How much load varied over the past X intervals |
| `rolling_max_X` | Highest load seen over the past X intervals |
| `rolling_min_X` | Lowest load seen over the past X intervals |

**Trend** — how load is changing over time

| Column | Description |
|--------|-------------|
| `delta_X` | Load change between X intervals ago and 1 interval ago |
| `slope_X` | Rate of change in load over the past X intervals (linear trend) |

`X` represents number of time interval (depending on the resolution)
- 1 min resolution: 1X = 1 minute, 2X = 2 minute, etc
- 5 min resolution: 1X = 5 minutes, 2X = 10 minutes, etc


In [ ]:
# ---------------------------------------------------------------
# Resample By Resolution
# ---------------------------------------------------------------
def resample_data(df: pd.DataFrame, resolution: str) -> pd.DataFrame:
    """Resample to target resolution by averaging load per interval."""
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    df = df["load"].resample(resolution).mean()
    df = df.reset_index()
    return df

# ---------------------------------------------------------------
# 1. Temporal Features
# ---------------------------------------------------------------
def time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    df["second"] = df["timestamp"].dt.second
    df["minute"] = df["timestamp"].dt.minute
    df["hour"] = df["timestamp"].dt.hour
    df["time_of_day"] = df["hour"].apply(time_of_day)
    df["day"] = df["timestamp"].dt.day
    df["weekday"] = df["timestamp"].dt.dayofweek
    df["is_weekend"] = df["timestamp"].dt.dayofweek.isin([5, 6]).astype(int)
    df["month"] = df["timestamp"].dt.month
    return df

# ---------------------------------------------------------------
# 2. Business Features
# ---------------------------------------------------------------
day_class_map = clean_df.groupby(clean_df["timestamp"].dt.date)["day_class"].first()

def add_business_features(df: pd.DataFrame) -> pd.DataFrame:
    df["workday"] = df["timestamp"].dt.date.map(day_class_map)
    return df

# ---------------------------------------------------------------
# 3. Historic Load Features
# ---------------------------------------------------------------
import pandas as pd

def add_lag_features(df: pd.DataFrame, intervals: dict, resolution: str) -> pd.DataFrame:
    
    df = df.copy()

    resolution_map = {"1min": 60, "5min": 300, "10min": 600}

    if resolution not in resolution_map:
        raise ValueError("Unsupported resolution")

    resolution_seconds = resolution_map[resolution]

    for name, minutes in intervals.items():
        
        lag_seconds = minutes * 60
        lag_steps = int(lag_seconds / resolution_seconds)

        df[f"lag_{name}"] = df["load"].shift(lag_steps)

    return df

def add_rolling_features(df: pd.DataFrame, windows: list) -> pd.DataFrame:
    for x in windows:
        df[f"rolling_mean_{x}"] = df["load"].shift(1).rolling(x).mean()
        df[f"rolling_std_{x}"]  = df["load"].shift(1).rolling(x).std()
        df[f"rolling_max_{x}"]  = df["load"].shift(1).rolling(x).max()
        df[f"rolling_min_{x}"]  = df["load"].shift(1).rolling(x).min()
    return df

def add_delta_features(df: pd.DataFrame, intervals: list) -> pd.DataFrame:
    for x in intervals:
        df[f"delta_{x}"] = df["load"].shift(1) - df["load"].shift(x)
    return df

def add_slope_features(df: pd.DataFrame, intervals: list) -> pd.DataFrame:
    def _slope(arr):
        if np.isnan(arr).any():
            return np.nan
        x = np.arange(len(arr))
        return np.polyfit(x, arr, 1)[0]

    for x in intervals:
        df[f"slope_{x}"] = (
            df["load"].shift(1).rolling(x).apply(_slope, raw=True)
        )
    return df

def add_historic_load_features(df: pd.DataFrame, res: str) -> pd.DataFrame:
    df = add_lag_features(df, LAG_INTERVALS, res)
    df = add_rolling_features(df, WINDOW_INTERVALS)
    df = add_delta_features(df, WINDOW_INTERVALS)
    df = add_slope_features(df, WINDOW_INTERVALS)

    return df

# ---------------------------------------------------------------
# Resample and Engineer Features by Resolution
# ---------------------------------------------------------------
dfs = {}

print("Resolution Datasets:")
for res in RESAMPLE_RESOLUTIONS:
    print(res, ":", end=" ")
    df = resample_data(clean_df, res)
    df = add_business_features(df)
    df = add_historic_load_features(df)
    df = add_temporal_features(df)
    dfs[res] = df
    print(df.shape)

Resolution Datasets:
1min : 

TypeError: add_lag_features() missing 1 required positional argument: 'resolution'

In [4]:
dfs["1min"].head()

,timestamp,load,workday,lag_1,lag_2,lag_3,lag_4,lag_5,lag_10,lag_15,...,slope_30,slope_60,second,minute,hour,time_of_day,day,weekday,is_weekend,month
0,2025-11-28 00:00:00,1912.400000,full,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,0,Night,28,4,0,11
1,2025-11-28 00:01:00,1908.233333,full,1912.400000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,1,0,Night,28,4,0,11
2,2025-11-28 00:02:00,1643.183333,full,1908.233333,1912.400000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,2,0,Night,28,4,0,11
3,2025-11-28 00:03:00,2992.950000,full,1643.183333,1908.233333,1912.400000,NaN,NaN,NaN,NaN,...,NaN,NaN,0,3,0,Night,28,4,0,11
4,2025-11-28 00:04:00,1106.000000,full,2992.950000,1643.183333,1908.233333,1912.4,NaN,NaN,NaN,...,NaN,NaN,0,4,0,Night,28,4,0,11


In [5]:
dfs["1min"].dtypes

timestamp          datetime64[ns]
load                      float64
workday                    object
lag_1                     float64
lag_2                     float64
lag_3                     float64
lag_4                     float64
lag_5                     float64
lag_10                    float64
lag_15                    float64
lag_30                    float64
lag_60                    float64
rolling_mean_5            float64
rolling_std_5             float64
rolling_max_5             float64
rolling_min_5             float64
rolling_mean_15           float64
rolling_std_15            float64
rolling_max_15            float64
rolling_min_15            float64
rolling_mean_30           float64
rolling_std_30            float64
rolling_max_30            float64
rolling_min_30            float64
rolling_mean_60           float64
rolling_std_60            float64
rolling_max_60            float64
rolling_min_60            float64
delta_5                   float64
delta_15      

In [8]:
# save each engineered dataset
for res, df in dfs.items():
    df.to_parquet(feature_engineered_filepath(res), index=False)